# AmazonML — retrain on the full per-country pool (Colab)

Runs the retrain **remotely** so it does not compete with the local test
inference for RAM.

Why this exists
---------------
The shipped model was trained against `val_sample`'s 518k pool while
India's test pool is 4.72M. 18 of the 53 features are channel / rank /
density counts, so those are only calibrated if training density resembles
test density. This run blocks the **full** per-country train pool.

Data comes from the public Drive folder, so no OAuth is needed. The code
is embedded below as a base64 tarball generated from the exact files
running locally, so it cannot drift from the local tree.

**Runtime: expect 2.5-3 h.** Colab free sessions can disconnect on idle;
if that happens, re-run and it will pick up from scratch.


## 1. Dependencies

In [ ]:
!pip -q install polars pandas numpy scipy scikit-learn gdown
# sparse-dot-topn compiles a C++ extension; Colab ships gcc.
!pip -q install sparse-dot-topn
!pip -q install xgboost lightgbm catboost
print("deps ok")

## 2. Unpack the code

In [ ]:
import base64, tarfile, io, os
B64 = (
    "..."
)
ROOT = "/content/AmazonML"
os.makedirs(ROOT, exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(B64)), mode="r:gz").extractall(ROOT)
os.chdir(ROOT)
print("unpacked ->", ROOT)
print(sorted(os.listdir(ROOT)))


## 3. Fetch the train split from Drive

Public folder, so `gdown` works with no credentials.

In [ ]:
!gdown --folder "https://drive.google.com/drive/folders/1BfqxOy-J6gzgUtiOurfx7GiOo9oe5hsg" -O /content/AmazonML/dataset 2>&1 | tail -5
!ls -la /content/AmazonML/dataset/train/

## 4. Retrain

`--sample-frac 0.08` is ~177k S1 entities out of 2.21M, stratified so the
singleton rate matches the full data (5.58%). `ER_TRAIN_CHUNK=5000` keeps
feature-extraction chunks small; the frame is cast to float32 per chunk so
the ~8M candidate pairs never all sit in float64.


In [ ]:
!ER_TRAIN_CHUNK=5000 ER_POOL_SHARD=1000000 ER_POOL_SHARD_CHAR=400000 \
  python -u train_model.py \
    --blocker sparse --max-candidates 45 --sample-frac 0.08 \
    --out /content/AmazonML/models_v2 \
    --verdict-file /content/AmazonML/logs/retrain_verdict.txt 2>&1 | tail -60

## 5. Results

Compare against the incumbent, which scored **96.6963%** OOF macro F0.5
on val_sample with blocking pair recall 98.11% / entity full coverage
94.10%.

The local driver gates on absolute OOF F0.5 >= 96.50%, so read the
`BEST rule=...` line above: at or above that, the retrained model is used
for the second test run; below it, the incumbent is kept.

If the number looks good, download `models_v2/` (it contains the three
boosters plus `decision_rule.json`, the thresholds inference adopts).


In [ ]:
import json, os, glob
d = "/content/AmazonML/models_v2"
print("files:", sorted(os.path.basename(f) for f in glob.glob(d + "/*")))
p = os.path.join(d, "decision_rule.json")
if os.path.exists(p):
    print(json.dumps(json.load(open(p)), indent=2))
v = "/content/AmazonML/logs/retrain_verdict.txt"
print(open(v).read() if os.path.exists(v) else "(no verdict file)")